# ShopSense 06: Evaluate search and answers

**Goal:** measure how good the system is, with numbers, using the golden set and your labels.

**Two parts**
1. **Retrieval:** for each labelled question, do BM25, vector search and hybrid put the relevant reviews in the top 5? We measure *hit rate*, *precision*, *MRR* and *nDCG* at 5.
2. **Answers:** run the answer step on the golden questions and measure: are citations valid, are sentences cited, does the system **refuse** when it should and **answer** when it should, and do cited reviews really contain the words of the claim?

**Reads:** `data/eval/golden_set.csv`, `data/eval/retrieval_labels.csv`.
**Writes:** result tables in `reports/eval/`.

**Needs:** Ollama running with `llama3.2:3b` (part 2 calls it about 40 times, roughly 10 to 20 minutes). Also put the file `answer.py` in `src/shopsense/llm/`.

### The four retrieval numbers, in plain words
| Metric | Meaning |
|---|---|
| Hit@5 | In how many questions is **at least one** relevant review in the top 5? |
| Precision@5 | Of the 5 results, what share is relevant? |
| MRR | How high is the **first** relevant result? (1 = first place, 0.5 = second, 0 = none) |
| nDCG@5 | Do relevant results come **early**? Like precision, but early positions count more. |

**Fair-test note:** your labelling sheet contained the top 5 of **every** method, so every result we look at in the top 5 has a label. That is why we stop at 5.

## 1. Setup

**What this does:** imports libraries, loads the search index (about 20 seconds), and loads the golden set and your labels.

In [1]:
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from shopsense.llm.answer import NO_ANSWER, answer_question
from shopsense.retrieval.hybrid import load_index, search

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

OUT = ROOT / "reports" / "eval"
OUT.mkdir(parents=True, exist_ok=True)

golden = pd.read_csv(ROOT / "data" / "eval" / "golden_set.csv")
labels = pd.read_csv(ROOT / "data" / "eval" / "retrieval_labels.csv")
index = load_index()
print("golden questions:", len(golden), "| labelled passages:", len(labels))

d:\Resume\Data Scientist\Resume_Projects\ShopSense_Product Review Intelligence Copilot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 495.12it/s]


golden questions: 67 | labelled passages: 196


**What this does:** checks the labels are complete (only 1 or 0, no blanks) and shows how many relevant passages each question has. Questions with **no** relevant passage cannot be scored for retrieval; they are tests of 'the reviews do not say'.

In [2]:
assert labels["relevant"].isin([0, 1]).all(), "relevant must be 1 or 0 on every row"

per_question = labels.groupby("question_id")["relevant"].agg(relevant_passages="sum", labelled="count")
print(per_question.T.to_string())

scorable = per_question[per_question["relevant_passages"] > 0].index.tolist()
no_evidence = per_question[per_question["relevant_passages"] == 0].index.tolist()
print("\nquestions scored for retrieval:", len(scorable))
print("questions with no relevant review (expected: refuse):", no_evidence)

question_id        G003  G004  G008  G009  G013  G014  G018  G019  G023  G024  G031  G032  G036  G037  G041  G042  G046  G047  G051  G052  G056  G057
relevant_passages     0     7     4     3     3     0     7     2     3     2     4     1     5     7     2     7     5     4     6     0     7     1
labelled              7     7     9     9     9    11     9    11     7     8     8     8    10     9     9     9    14    10     8     8     8     8

questions scored for retrieval: 19
questions with no relevant review (expected: refuse): ['G003', 'G014', 'G052']


## 2. Retrieval: BM25 vs vector vs hybrid

**What this does:** for every scorable question and every method, takes the top 5 results (searching only inside that question's product) and looks up each result's label. Then it computes the four numbers.

Results that have no label would be a problem (we would not know if they are relevant). The cell counts them and should print 0.

In [3]:
is_relevant = {}
for row in labels.itertuples():
    is_relevant[(row.question_id, row.passage_id)] = int(row.relevant)

TOP = 5
results = []
unlabelled = 0

for qid in scorable:
    question = golden.loc[golden["id"] == qid].iloc[0]
    total_relevant = int(per_question.loc[qid, "relevant_passages"])

    for method in ["bm25", "vector", "hybrid"]:
        found = search(index, question["question"], asin=question["product_asin"], top_k=TOP, method=method)

        flags = []
        for passage_id in found["passage_id"]:
            key = (qid, passage_id)
            if key not in is_relevant:
                unlabelled += 1
            flags.append(is_relevant.get(key, 0))

        hit = 1 if sum(flags) > 0 else 0
        precision = sum(flags) / TOP
        first = [i for i, f in enumerate(flags) if f == 1]
        mrr = 1 / (first[0] + 1) if first else 0

        dcg = sum(f / np.log2(i + 2) for i, f in enumerate(flags))
        ideal = sum(1 / np.log2(i + 2) for i in range(min(total_relevant, TOP)))
        ndcg = dcg / ideal if ideal > 0 else 0

        results.append({"question_id": qid, "method": method, "hit@5": hit, "precision@5": precision, "mrr": mrr, "ndcg@5": ndcg})

retrieval = pd.DataFrame(results)
print("results without a label (should be 0):", unlabelled)
retrieval.head(6)

results without a label (should be 0): 0


,question_id,method,hit@5,precision@5,mrr,ndcg@5
0,G004,bm25,1,1.0,1.0,1.000000
1,G004,vector,1,1.0,1.0,1.000000
2,G004,hybrid,1,1.0,1.0,1.000000
3,G008,bm25,1,0.2,1.0,0.390380
4,G008,vector,1,0.6,0.5,0.592512
5,G008,hybrid,1,0.4,0.5,0.397322


**What this does:** the main table: average of each metric per method.

In [4]:
summary = retrieval.groupby("method")[["hit@5", "precision@5", "mrr", "ndcg@5"]].mean().round(3)
summary = summary.loc[["bm25", "vector", "hybrid"]]
summary.to_csv(OUT / "06_retrieval_summary.csv")
retrieval.to_csv(OUT / "06_retrieval_by_question.csv", index=False)
summary

,hit@5,precision@5,mrr,ndcg@5
method,,,,
bm25,0.842,0.421,0.607,0.477
vector,1.000,0.558,0.704,0.673
hybrid,0.842,0.484,0.728,0.582


**What this does:** asks whether hybrid is *really* better or just lucky. With only about 19 questions, differences can be noise. We **bootstrap**: re-draw the questions with replacement 2,000 times and see how much the difference (hybrid minus the other method) moves. If the 95% range includes 0, we cannot claim a difference.

In [5]:
rng = np.random.default_rng(42)
wide = retrieval.pivot(index="question_id", columns="method", values="ndcg@5")
n = len(wide)

rows = []
for other in ["bm25", "vector"]:
    differences = []
    for _ in range(2000):
        picked = rng.integers(0, n, n)
        sample = wide.iloc[picked]
        differences.append((sample["hybrid"] - sample[other]).mean())
    low, high = np.percentile(differences, [2.5, 97.5])
    rows.append({"comparison": f"hybrid minus {other} (nDCG@5)", "mean_difference": round((wide["hybrid"] - wide[other]).mean(), 3),
                 "ci_low": round(low, 3), "ci_high": round(high, 3), "clearly_better": low > 0})

pd.DataFrame(rows)

,comparison,mean_difference,ci_low,ci_high,clearly_better
0,hybrid minus bm25 (nDCG@5),0.104,0.010,0.203,True
1,hybrid minus vector (nDCG@5),-0.091,-0.219,0.044,False


**What this does:** shows which questions each method fails on (Hit@5 = 0). Read these: they tell you what to fix.

In [6]:
hits_wide = retrieval.pivot(index="question_id", columns="method", values="hit@5")
failed = hits_wide[(hits_wide == 0).any(axis=1)]
print("questions where at least one method found nothing relevant:")
failed.join(golden.set_index("id")["question"].str.slice(0, 90))

questions where at least one method found nothing relevant:


,bm25,hybrid,vector,question
question_id,,,,
G019,1,0,1,What do reviewers say about the packaging of Murray & Lanman Florida Water Plastic Bottle
G024,0,1,1,What do reviewers say about the packaging of Verb Ghost Shampoo & Conditioner Duo – Vegan
G032,0,0,1,What do reviewers say about the value for money of Yeshan bow Headbands for women Non Slip
G057,0,0,1,What do reviewers say about the skin reaction of Lume Natural Deodorant - Underarms and Pr


## 3. Answers: run the golden questions through the full system

**What this does:** picks the questions for the answer test:
- `aspect_summary` questions (should be **answered** with citations; 3 of them have no relevant reviews, so a refusal is acceptable there),
- `unanswerable` and `out_of_scope` questions (should be **refused**).

Numeric and factual questions need a calculator tool, which we add with the agent later. Comparison questions need two products, also for the agent.

In [7]:
test_questions = golden[golden["question_type"].isin(["aspect_summary", "unanswerable", "out_of_scope"])].copy()
print(test_questions["question_type"].value_counts().to_string())
print("calls to the model:", len(test_questions), "(about 10 to 20 minutes)")

question_type
aspect_summary    22
unanswerable      12
out_of_scope       5
calls to the model: 39 (about 10 to 20 minutes)


**What this does:** runs every question. Each result is saved to disk as it finishes, so a crash does not lose everything. Progress is printed.

In [8]:
rows = []
for number, q in enumerate(test_questions.itertuples(), start=1):
    asin = q.product_asin if isinstance(q.product_asin, str) and q.product_asin != "" else None
    result = answer_question(index, q.question, asin=asin)

    rows.append({"id": q.id, "type": q.question_type, "question": q.question, "answer": result["answer"],
                 "refused": result["refused"], "cited": str(result["cited"]), "invalid": str(result["invalid"]),
                 "uncited_sentences": result["uncited"], "seconds": result["seconds"], "tokens": result["tokens"],
                 "reason": result["reason"],
                 "passages": " || ".join(result["hits"]["text"].str.slice(0, 400).tolist()) if result["hits"] is not None else ""})
    pd.DataFrame(rows).to_csv(OUT / "06_answers.csv", index=False)
    print(f"{number}/{len(test_questions)} {q.id} {q.question_type:15s} refused={result['refused']} {result['seconds']:.0f}s")

answers = pd.DataFrame(rows)

1/39 G003 aspect_summary  refused=False 102s
2/39 G004 aspect_summary  refused=False 24s
3/39 G005 unanswerable    refused=True 14s
4/39 G008 aspect_summary  refused=False 37s
5/39 G009 aspect_summary  refused=False 32s
6/39 G010 unanswerable    refused=True 12s
7/39 G013 aspect_summary  refused=False 36s
8/39 G014 aspect_summary  refused=False 18s
9/39 G015 unanswerable    refused=True 13s
10/39 G018 aspect_summary  refused=False 40s
11/39 G019 aspect_summary  refused=False 16s
12/39 G020 unanswerable    refused=True 11s
13/39 G023 aspect_summary  refused=False 32s
14/39 G024 aspect_summary  refused=False 31s
15/39 G025 unanswerable    refused=False 27s
16/39 G028 unanswerable    refused=True 14s
17/39 G031 aspect_summary  refused=False 35s
18/39 G032 aspect_summary  refused=False 32s
19/39 G033 unanswerable    refused=True 8s
20/39 G036 aspect_summary  refused=False 32s
21/39 G037 aspect_summary  refused=False 25s
22/39 G038 unanswerable    refused=True 11s
23/39 G041 aspect_summary 

## 4. Score the answers

**What this does:** measures, per question type:
- **Refusal rate**: for `unanswerable` and `out_of_scope` questions this should be near 100%. For `aspect_summary` questions that have relevant reviews it should be low (a refusal there is a **false refusal**).
- **Invalid citations**: citing `[n]` that does not exist. Should be 0.
- **Uncited sentences**: sentences with no `[n]`. Should be 0.

In [9]:
answers["has_evidence"] = ~answers["id"].isin(no_evidence)
answers["invalid_count"] = answers["invalid"].str.count(r"\d+")

for kind in ["aspect_summary", "unanswerable", "out_of_scope"]:
    part = answers[answers["type"] == kind]
    print(f"{kind:15s} questions={len(part):3d} refused={100 * part['refused'].mean():5.1f}%")

aspect = answers[answers["type"] == "aspect_summary"]
with_evidence = aspect[aspect["has_evidence"]]
print()
print("aspect questions WITH evidence that were wrongly refused:", int(with_evidence["refused"].sum()), "of", len(with_evidence))
answered = answers[~answers["refused"]]
print("answers with an invalid citation :", int((answered["invalid_count"] > 0).sum()), "of", len(answered))
print("answers with an uncited sentence :", int((answered["uncited_sentences"] > 0).sum()), "of", len(answered))
print("average seconds per answer       :", round(answers["seconds"].mean(), 1))

aspect_summary  questions= 22 refused=  0.0%
unanswerable    questions= 12 refused= 91.7%
out_of_scope    questions=  5 refused= 80.0%

aspect questions WITH evidence that were wrongly refused: 0 of 19
answers with an invalid citation : 0 of 24
answers with an uncited sentence : 12 of 24
average seconds per answer       : 27.8


**What this does:** a rough **support check**. For each cited sentence, what share of its content words (4+ letters, not common words) appear in the passages it cites? A low share means the sentence may not come from the review. This is a crude test (it cannot see paraphrases), so use it only to **find answers to read by hand**, not as a final score.

In [10]:
STOP = set("this that with from they have were been their there about which would could should what when your very more than also just like only some into them then these those because while after before other".split())
passage_lists = {row.id: row.passages.split(" || ") for row in answers.itertuples()}

support_rows = []
for row in answers[~answers["refused"]].itertuples():
    passages = passage_lists[row.id]
    for sentence in re.split(r"(?<=[.!?])\s+", row.answer):
        numbers = [int(n) for n in re.findall(r"\[(\d+)\]", sentence)]
        numbers = [n for n in numbers if 1 <= n <= len(passages)]
        if not numbers:
            continue
        words = [w for w in re.findall(r"[a-z]{4,}", re.sub(r"\[\d+\]", "", sentence).lower()) if w not in STOP]
        source = " ".join(passages[n - 1] for n in numbers).lower()
        if words:
            share = sum(1 for w in words if w in source) / len(words)
            support_rows.append({"id": row.id, "support": round(share, 2), "sentence": sentence[:140]})

support = pd.DataFrame(support_rows)
if len(support):
    print("cited sentences checked:", len(support))
    print("average support share  :", round(support["support"].mean(), 2))
    print("sentences with support < 0.30 (read these):", int((support["support"] < 0.30).sum()))
    support.sort_values("support").head(8)
else:
    print("no cited sentences to check")

cited sentences checked: 69
average support share  : 0.3
sentences with support < 0.30 (read these): 34


## 5. Save

**What this does:** saves the scored answers and the numbers, so the README can quote them.

In [11]:
answers.drop(columns=["passages"]).to_csv(OUT / "06_answers_scored.csv", index=False)
if len(support):
    support.to_csv(OUT / "06_support_check.csv", index=False)

final = {
    "retrieval_questions_scored": len(scorable),
    "answer_questions": len(answers),
    "refusal_rate_unanswerable_pct": round(100 * answers[answers["type"] == "unanswerable"]["refused"].mean(), 1),
    "refusal_rate_out_of_scope_pct": round(100 * answers[answers["type"] == "out_of_scope"]["refused"].mean(), 1),
    "false_refusals_with_evidence": int(with_evidence["refused"].sum()),
    "answers_with_invalid_citation": int((answered["invalid_count"] > 0).sum()),
    "avg_seconds_per_answer": round(answers["seconds"].mean(), 1),
}
pd.Series(final)

retrieval_questions_scored       19.0
answer_questions                 39.0
refusal_rate_unanswerable_pct    91.7
refusal_rate_out_of_scope_pct    80.0
false_refusals_with_evidence      0.0
answers_with_invalid_citation     0.0
avg_seconds_per_answer           27.8
dtype: float64

## Read these by hand (15 minutes)

Open `reports/eval/06_answers.csv`. For **10 answers** (mix of types), check: is each claim in the cited passage? Note the ones that are not. These real examples are your best interview material, and they tell us what to fix before building the agent.

## How to explain this in an interview

- "I evaluate retrieval and generation **separately**. If the right review is not in the top 5, no language model can fix it, so I measure retrieval first."
- "I built a golden set and labelled relevance for the top 5 of every method pooled together, so the comparison is fair. Metrics are Hit@5, precision, MRR and nDCG@5. I bootstrapped the differences, and with about 19 scorable questions I report the interval, because small samples overstate differences."
- "For answers I measured: valid citations, uncited sentences, refusal on impossible or out-of-scope questions, and **false refusals**. The two refusal errors have different costs, so I track them separately."
- "The word-overlap support check is a rough screen to find answers to read, not a score. I read a sample by hand and report what I found."
- "Limits: the golden set is small, I used one 3B model, and labels were AI-drafted then reviewed by me, so I treat the numbers as a baseline to improve, not a final claim."